# CeNN V5.2 — Qwen3.5-0.8B | Fidelity Recovery & Controlled Head-Rescue

**Research Colab · [TinyCeNN-LM](https://github.com/vtavakkoli/TinyCeNN-LM) · standalone no-upload execution**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Qwen35_08B_CeNN_HeadRescue_V52_FidelityRecovery_Colab.ipynb)

**Objective:** Improve on the observed V5.1 layer-11 run without misreporting benchmark wins. That run achieved validation cosine 0.94984 for router-only and 0.95748 for joint training, but agreement near 97% and **audit speedup 0.953×**; no candidate passed. V5.2 tests a causal, output-conditioned low-rank memory correction, longer local windows and automatic optional model-logit KL alignment.

This is **not** a published faster Qwen replacement. Original model, exact KV group 0, projections, norms, RoPE and feed-forward blocks are preserved. Only approximate KV group 1 is trained. CPU contract tests precede CUDA training, and the quality/speed acceptance gate is strict. **No manual checkpoint or dataset upload required.**


## 0 · Environment and reproducibility

Select **Runtime → Change runtime type → GPU**, then **Runtime → Run all**. This notebook clones the public repository and imports canonical V5/V5.1/V5.2 modules. It downloads Qwen and WikiText-2, trains an automatic compatible V5 baseline, and exports an archive. Training state is preserved within the current Colab filesystem; a fresh runtime will retrain, not magically restore a remote checkpoint.


In [ ]:
%pip -q install "transformers==5.17.0" "huggingface_hub>=0.36,<2" "datasets>=3,<5" "accelerate>=1" "safetensors" "pandas>=2" "matplotlib>=3.7" "pytest>=8"

In [ ]:
import io, os, sys, gc, json, hashlib, subprocess, math, random, statistics, time
from pathlib import Path
from contextlib import contextmanager, nullcontext
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

REPO=Path('/content/TinyCeNN-LM')
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','--quiet','--depth','1','--branch','main',
                    'https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','--quiet','origin','main'],check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'--no-deps'],check=True)
sys.path.insert(0,str(REPO/'src'))

# Fail fast on missing repository source rather than creating an untracked fork.
V51_MODULE=REPO/'src/tinycenn_lm/cenn_head_rescue_v51.py'
V51_TESTS=REPO/'tests/test_cenn_head_rescue_v51.py'
V52_MODULE=REPO/'src/tinycenn_lm/cenn_head_rescue_v52.py'
V52_TESTS=REPO/'tests/test_cenn_head_rescue_v52.py'
assert all(p.is_file() for p in (V51_MODULE,V51_TESTS,V52_MODULE,V52_TESTS)), (
    'V5.1 module/tests missing from cloned repository. Open the published main-branch notebook.')
MODULE_SHA256=hashlib.sha256(V51_MODULE.read_bytes()+V52_MODULE.read_bytes()).hexdigest()
from tinycenn_lm.cenn_deltacell_v3 import CeNNDeltaCellV3
from tinycenn_lm.cenn_head_rescue_v5 import Qwen35HeadRescueV5
from tinycenn_lm.cenn_head_rescue_v51 import CeNNQueryRescueV51,warmstart_v5_delta_core
from tinycenn_lm.cenn_head_rescue_v52 import CeNNHeadRescueV52,warmstart_v51_to_v52
from transformers import AutoTokenizer, AutoConfig, Qwen3_5ForCausalLM
from huggingface_hub import model_info
GIT_SHA=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert torch.cuda.is_available(), 'This full Qwen experiment requires an NVIDIA CUDA runtime.'
DEVICE=torch.device('cuda')
DTYPE=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print('Git commit',GIT_SHA,'V5.1+V5.2 source SHA256',MODULE_SHA256)
print('GPU:',torch.cuda.get_device_name(0),'dtype:',DTYPE)
subprocess.run([sys.executable,'-m','pytest','-q',str(V51_TESTS),str(V52_TESTS)],check=True)


## 1 · Experiment controls

Six candidates receive **identical automatic V5 baseline weights**, with the same training sequence sampling and optimizer budget. Window-128 candidates trade additional compute/state for a wider exact causal local memory; window-64 candidates provide a matched control. V5.2 starts with a zero post-correction and matches the original V5.1 behavior at window 64.


In [ ]:
# @title No-upload experiment settings
PROFILE='quick' # @param ['quick','research']
MODEL_ID='Qwen/Qwen3.5-0.8B'
TARGET_LAYER=11
SEQ_LEN=256 if PROFILE=='quick' else 512
TRAIN_BLOCKS=48 if PROFILE=='quick' else 160
DEV_BLOCKS=24 if PROFILE=='quick' else 64
AUDIT_BLOCKS=24 if PROFILE=='quick' else 64
AUDIT_START_TOKEN=1_000_000
BASELINE_STEPS=140 if PROFILE=='quick' else 600
STEPS=220 if PROFILE=='quick' else 900
TASK_STEPS=25 if PROFILE=='quick' else 100
BATCH_SIZE=2
BASELINE_LR=2e-3
LR=8e-4
CHUNK_SIZE=32
SEED=3051
RANK=16
OUT=Path('/content/qwen35_cenn_head_rescue_v52')
OUT.mkdir(parents=True,exist_ok=True)
(OUT/'checkpoints').mkdir(exist_ok=True)
CANDIDATES={
 'v5_frozen':dict(family='v5',router=False,correction=False,post=False,window=64,train='none'),
 'v51_router_w64':dict(family='v51',router=True,correction=False,post=False,window=64,train='router'),
 'v52_post_w64':dict(family='v52',router=False,correction=False,post=True,window=64,train='post'),
 'v52_router_post_w64':dict(family='v52',router=True,correction=False,post=True,window=64,train='router_post'),
 'v52_router_post_w128':dict(family='v52',router=True,correction=False,post=True,window=128,train='router_post'),
 'v52_joint_w128':dict(family='v52',router=True,correction=True,post=True,window=128,train='joint'),
}

random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32=True
print('Automatic V5 baseline:', BASELINE_STEPS, 'steps; V5.1:', STEPS,
      'steps per trainable candidate; no upload required')


## 2. Load the **version-pinned** original Qwen model
Keep the original frozen throughout. Layer 11 is Qwen3.5 **full attention**, not Gated DeltaNet. Before experiments, verify an all-exact V5 adapter reproduces native Qwen logits.


In [ ]:
REVISION=model_info(MODEL_ID).sha
print('Pinned Qwen model revision:',REVISION)
cfg=AutoConfig.from_pretrained(MODEL_ID,revision=REVISION).get_text_config(decoder=True)
assert cfg.model_type=='qwen3_5_text' and cfg.num_hidden_layers==24 and cfg.hidden_size==1024
full=[i for i,t in enumerate(cfg.layer_types) if t=='full_attention']
assert TARGET_LAYER in full and full==[3,7,11,15,19,23],full
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION,use_fast=True)
model=Qwen3_5ForCausalLM.from_pretrained(MODEL_ID,revision=REVISION,dtype=DTYPE,
    attn_implementation='sdpa',low_cpu_mem_usage=True).to(DEVICE).eval()
model.requires_grad_(False)
model.config.use_cache=False
LAYERS=model.model.layers
NATIVE=LAYERS[TARGET_LAYER].self_attn
HD=int(NATIVE.head_dim)
QHEADS=int(NATIVE.q_proj.out_features//(2*HD))
KVHEADS=int(NATIVE.k_proj.out_features//HD)
assert (QHEADS,KVHEADS,HD)==(8,2,256),(QHEADS,KVHEADS,HD)
@contextmanager
def graft(adapter):
    assert LAYERS[TARGET_LAYER].self_attn is NATIVE
    LAYERS[TARGET_LAYER].self_attn=adapter
    try:yield
    finally:LAYERS[TARGET_LAYER].self_attn=NATIVE
sample=tokenizer('CeNN is a cellular associative memory model.',return_tensors='pt',
                 add_special_tokens=False).input_ids.to(DEVICE)
all_exact=Qwen35HeadRescueV5(NATIVE,None,exact_groups=(0,1)).eval()
with torch.inference_mode(),torch.autocast('cuda',dtype=DTYPE):
    reference=model(input_ids=sample,use_cache=False,logits_to_keep=0).logits.float()
    with graft(all_exact):parity=model(input_ids=sample,use_cache=False,logits_to_keep=0).logits.float()
max_difference=(reference-parity).abs().max().item()
print('Exact-native parity:',max_difference)
assert max_difference<.05 and bool((reference.argmax(-1)==parity.argmax(-1)).all())
del all_exact,parity,reference


## 3 · Calibration, validation and fixed diagnostic partition

Fit the baseline and all V5.1 candidates using the **same training prefix**. Select by WikiText-2 `validation`. The predeclared diagnostic window comes from the distant `train` tail after token 250,000; it is withheld from calibration/selection but is **not an independent external dataset**.


In [ ]:
from datasets import load_dataset

def build_windows(split,count,length,start_token=0):
    ds=load_dataset('Salesforce/wikitext','wikitext-2-raw-v1',split=split)
    need=start_token+count*length
    tokens=[]
    for row in ds:
        text=row['text'].strip()
        if not text:continue
        tokens.extend(tokenizer.encode(text,add_special_tokens=False))
        tokens.append(tokenizer.eos_token_id)
        if len(tokens)>=need:break
    if len(tokens)<need:raise RuntimeError(f'{split}: {len(tokens)} tokens, need {need}')
    return torch.tensor(tokens[start_token:need],dtype=torch.long).view(count,length)
TRAIN=build_windows('train',TRAIN_BLOCKS,SEQ_LEN)
DEV=build_windows('validation',DEV_BLOCKS,SEQ_LEN)
# This split is not constructed before model selection. AUDIT_START_TOKEN is fixed above.
FINGERPRINTS={'train':hashlib.sha256(TRAIN.numpy().tobytes()).hexdigest(),
              'validation':hashlib.sha256(DEV.numpy().tobytes()).hexdigest()}
assert TRAIN_BLOCKS*SEQ_LEN < AUDIT_START_TOKEN
print('Train',TRAIN.shape,'Dev',DEV.shape,'Audit offset',AUDIT_START_TOKEN)

## 4. Capture **original teacher mixer** inputs and outputs once
Hook the actual full-attention module, retaining Qwen's RoPE tensors. None of this includes the previously held-out audit texts.

In [ ]:
TEACHER={}
@torch.inference_mode()
def forward(ids,keep=1):
    with torch.autocast('cuda',dtype=DTYPE):
        return model(input_ids=ids.to(DEVICE),use_cache=False,logits_to_keep=keep)

def capture(name,blocks):
    cols={v:[] for v in ('x','y','cos','sin')}
    def hook(module,args,kwargs,result):
        inp=kwargs.get('hidden_states',args[0] if args else None)
        assert inp is not None
        val=result[0] if isinstance(result,tuple) else result
        cos,sin=kwargs['position_embeddings']
        cols['x'].append(inp.detach().to('cpu',dtype=torch.float16))
        cols['y'].append(val.detach().to('cpu',dtype=torch.float16))
        cols['cos'].append(cos.detach().cpu().to(torch.float32))
        cols['sin'].append(sin.detach().cpu().to(torch.float32))
    handle=NATIVE.register_forward_hook(hook,with_kwargs=True)
    try:
        for start in range(0,len(blocks),BATCH_SIZE):forward(blocks[start:start+BATCH_SIZE])
    finally:handle.remove()
    TEACHER[name]={key:torch.cat(val,0).contiguous() for key,val in cols.items()}
    print(name,{key:tuple(val.shape) for key,val in TEACHER[name].items()})
capture('train',TRAIN)
capture('dev',DEV)
torch.cuda.empty_cache()

## 5 · Construct comparable V5.2 candidates

V5.2 loads every existing V5.1 tensor from the same automatically trained V5 baseline, and introduces only three new post-projection parameters. **The post up-projection is zero initialized.** Window 128 is an ablation, not a parity-preserving replacement of window 64 for long sequences.


In [ ]:
def teacher_batch(split,indices):
    raw=TEACHER[split]
    return (raw['x'][indices].to(DEVICE,dtype=DTYPE),
            raw['y'][indices].to(DEVICE,dtype=DTYPE),
            (raw['cos'][indices].to(DEVICE),raw['sin'][indices].to(DEVICE)))

def local_metrics(pred,target):
    a,b=pred.float(),target.float()
    return ((a-b).square().mean()/b.square().mean().clamp_min(1e-8),
            F.cosine_similarity(a,b,dim=-1).mean())

def new_v5_baseline():
    core=CeNNDeltaCellV3(4,1,HD,feature_dim=64,chunk_size=CHUNK_SIZE,
                         mode='none',local_window=64).to(DEVICE)
    return Qwen35HeadRescueV5(NATIVE,core,exact_groups=(0,)).eval()

@torch.no_grad()
def validate_baseline(adapter):
    adapter.eval()
    error=energy=cosine_sum=0.;n=0
    for begin in range(0,DEV_BLOCKS,BATCH_SIZE):
        indices=torch.arange(begin,min(begin+BATCH_SIZE,DEV_BLOCKS))
        x,y,rope=teacher_batch('dev',indices)
        with torch.autocast('cuda',dtype=DTYPE):
            pred,_=adapter(x,position_embeddings=rope)
        a,b=pred.float(),y.float()
        error+=(a-b).square().sum().item()
        energy+=b.square().sum().item()
        cosine_sum+=F.cosine_similarity(a,b,dim=-1).sum().item()
        n+=a.shape[0]*a.shape[1]
    return dict(nmse=error/max(energy,1.e-8),cosine=cosine_sum/n)

BOOTSTRAP_SPEC=dict(model_id=MODEL_ID,model_revision=REVISION,source_git_sha=GIT_SHA,
    source_module_sha256=MODULE_SHA256,layer=TARGET_LAYER,profile=PROFILE,
    train_hash=FINGERPRINTS['train'],dev_hash=FINGERPRINTS['validation'],
    feature_dim=64,local_window=64,seq_len=SEQ_LEN,train_blocks=TRAIN_BLOCKS,
    dev_blocks=DEV_BLOCKS,chunk_size=CHUNK_SIZE,baseline_steps=BASELINE_STEPS,
    batch_size=BATCH_SIZE,lr=BASELINE_LR,seed=SEED)
BOOTSTRAP_FINGERPRINT=hashlib.sha256(
    json.dumps(BOOTSTRAP_SPEC,sort_keys=True).encode()).hexdigest()
BOOTSTRAP_PATH=OUT/'checkpoints'/f'v5_auto_L{TARGET_LAYER}_delta_f64_w64.pt'

if BOOTSTRAP_PATH.exists():
    candidate=torch.load(BOOTSTRAP_PATH,map_location='cpu',weights_only=True)
    if candidate.get('bootstrap_fingerprint')==BOOTSTRAP_FINGERPRINT:
        baseline=new_v5_baseline()
        baseline.core.load_state_dict(candidate['core_state'],strict=True)
        V5_CORE_STATE={k:v.detach().cpu().clone() for k,v in candidate['core_state'].items()}
        BOOTSTRAP_VALIDATION=candidate['validation']
        del baseline
        print('Reusing locally trained baseline, matching fingerprint',BOOTSTRAP_FINGERPRINT[:12])
    else:
        print('Existing baseline settings differ; training a new baseline automatically')
        V5_CORE_STATE=None
else:
    V5_CORE_STATE=None

if V5_CORE_STATE is None:
    torch.manual_seed(SEED+TARGET_LAYER*17)
    baseline=new_v5_baseline()
    baseline.core.requires_grad_(True)
    optimizer=torch.optim.AdamW(baseline.core.parameters(),lr=BASELINE_LR,
                                weight_decay=.01)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,T_max=BASELINE_STEPS,eta_min=BASELINE_LR*.10)
    scaler=torch.amp.GradScaler('cuda',enabled=(DTYPE==torch.float16))
    sampling=torch.randint(0,TRAIN_BLOCKS,(BASELINE_STEPS,BATCH_SIZE),
        generator=torch.Generator().manual_seed(SEED+11))
    best=float('inf');best_sd=None;best_stats=None
    for step in range(1,BASELINE_STEPS+1):
        x,y,rope=teacher_batch('train',sampling[step-1])
        baseline.train();optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=DTYPE):
            pred,_=baseline(x,position_embeddings=rope)
            nmse,cosine=local_metrics(pred,y)
            loss=nmse+.2*(1.-cosine)
        if not bool(torch.isfinite(loss)):
            raise FloatingPointError(f'Nonfinite V5 baseline loss at step {step}')
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(baseline.core.parameters(),1.)
        scaler.step(optimizer);scaler.update();scheduler.step()
        if step==1 or step==BASELINE_STEPS or step%max(1,BASELINE_STEPS//8)==0:
            stats=validate_baseline(baseline)
            score=stats['nmse']+.2*(1.-stats['cosine'])
            if score<best:
                best=score;best_stats=stats
                best_sd={k:v.detach().cpu().clone()
                         for k,v in baseline.core.state_dict().items()}
            print(f'V5 auto-bootstrap {step}/{BASELINE_STEPS} '
                  f'cos={stats["cosine"]:.5f}, NMSE={stats["nmse"]:.5f}',flush=True)
    assert best_sd is not None
    V5_CORE_STATE=best_sd
    BOOTSTRAP_VALIDATION=best_stats
    torch.save(dict(core_state=best_sd,validation=best_stats,
                    bootstrap_fingerprint=BOOTSTRAP_FINGERPRINT,bootstrap_spec=BOOTSTRAP_SPEC,
                    model_revision=REVISION,variant='rescue_g0_delta_f64_w64'),
               BOOTSTRAP_PATH)
    del baseline,optimizer,scheduler,scaler,best_sd
    torch.cuda.empty_cache()
V5_CHECKPOINT_SHA256=hashlib.sha256(BOOTSTRAP_PATH.read_bytes()).hexdigest()
BOOTSTRAP_VARIANT='rescue_g0_delta_f64_w64'
assert len(V5_CORE_STATE)==len(new_v5_baseline().core.state_dict())
print('Baseline checkpoint:',BOOTSTRAP_PATH,'SHA-256:',V5_CHECKPOINT_SHA256)
print('Baseline mixer fidelity:', BOOTSTRAP_VALIDATION)


## 6 · Train V5.1 candidates from the auto-bootstrap baseline

Each candidate receives the same newly trained 64-feature Delta + Local64 weights. The five controlled ablations are a frozen baseline, router-only, correction-only, router+correction, and joint fine-tuning. No file upload is necessary.


In [ ]:
def base_variant(name):
    return 'v52_joint_w128' if name=='v52_joint_w128_plus_kl' else name

def build_candidate(name):
    base=base_variant(name)
    cfg=CANDIDATES[base]
    if cfg['family']=='v5':
        core=CeNNDeltaCellV3(4,1,HD,feature_dim=64,chunk_size=CHUNK_SIZE,
                             mode='none',local_window=64)
        core.load_state_dict(V5_CORE_STATE,strict=True)
    else:
        old=CeNNQueryRescueV51(4,1,HD,feature_dim=64,chunk_size=CHUNK_SIZE,
                               local_window=cfg['window'],correction_rank=RANK,
                               query_router=cfg['router'],query_correction=cfg['correction'])
        warmstart_v5_delta_core(old,V5_CORE_STATE)
        if cfg['family']=='v51':
            core=old
        else:
            core=CeNNHeadRescueV52(4,1,HD,feature_dim=64,chunk_size=CHUNK_SIZE,
                  local_window=cfg['window'],correction_rank=RANK,
                  query_router=cfg['router'],query_correction=cfg['correction'],
                  post_rank=RANK,post_correction=cfg['post'])
            warmstart_v51_to_v52(core,old.state_dict())
    core=core.to(DEVICE)
    return Qwen35HeadRescueV5(NATIVE,core,exact_groups=(0,)).eval()

with torch.inference_mode():
    x,y,rope=teacher_batch('dev',torch.tensor([0]))
    with torch.autocast('cuda',dtype=DTYPE):
        orig,_=build_candidate('v5_frozen')(x,position_embeddings=rope)
        v52,_=build_candidate('v52_post_w64')(x,position_embeddings=rope)
    parity=float((orig-v52).abs().max().item())
assert parity<0.01,parity
print('V5 → V5.2 zero-post warm-start max output difference:',parity)
del x,y,rope,orig,v52


## 6 · Local teacher calibration

Every candidate uses the same sampled batches and training length. Optimization minimizes normalized mixer MSE plus cosine distance. Evaluate all validation blocks periodically, retaining the best validation checkpoint. The full Qwen teacher stays frozen. At the next stage, separate language-model NLL/agreement/latency measurements determine whether local fidelity transfers to model-level behavior.


In [ ]:
def set_trainables(adapter,variant):
    core=adapter.core
    core.requires_grad_(False)
    mode=CANDIDATES[base_variant(variant)]['train']
    if mode=='none':return []
    router=['local_logits','router_w','router_bias']
    correction=['corr_down','corr_up','corr_log_gain']
    post=['post_down','post_up','post_log_gain']
    if mode=='router':names=router
    elif mode=='post':names=post
    elif mode=='router_post':names=router+post
    elif mode=='joint':names=[name for name,_ in core.named_parameters()]
    else:raise ValueError(mode)
    for name,parameter in core.named_parameters():
        if name in names:parameter.requires_grad_(True)
    enabled=[p for p in core.parameters() if p.requires_grad]
    assert enabled and not any(p.requires_grad for p in NATIVE.parameters())
    return enabled

@torch.no_grad()
def validate_local(adapter):
    adapter.eval();err=energy=cos_sum=0.;count=0
    for begin in range(0,DEV_BLOCKS,BATCH_SIZE):
        inds=torch.arange(begin,min(DEV_BLOCKS,begin+BATCH_SIZE))
        x,y,rope=teacher_batch('dev',inds)
        with torch.autocast('cuda',dtype=DTYPE):out,_=adapter(x,position_embeddings=rope)
        err+=(out.float()-y.float()).square().sum().item()
        energy+=y.float().square().sum().item()
        cos_sum+=F.cosine_similarity(out.float(),y.float(),dim=-1).sum().item()
        count+=out.shape[0]*out.shape[1]
    return dict(cosine=cos_sum/count,nmse=err/max(energy,1e-8))

def fit(variant):
    torch.manual_seed(SEED)
    adapter=build_candidate(variant)
    params=set_trainables(adapter,variant)
    if not params:
        return adapter,validate_local(adapter),{k:v.detach().cpu().clone() for k,v in adapter.core.state_dict().items()}
    opt=torch.optim.AdamW(params,lr=LR,weight_decay=.005)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=STEPS,eta_min=LR*.12)
    scaler=torch.amp.GradScaler('cuda',enabled=(DTYPE==torch.float16))
    best_loss=float('inf');best_sd=None;best_report=None
    indices=torch.randint(0,TRAIN_BLOCKS,(STEPS,BATCH_SIZE),generator=torch.Generator().manual_seed(SEED))
    for step in range(1,STEPS+1):
        x,y,rope=teacher_batch('train',indices[step-1])
        adapter.train();opt.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=DTYPE):
            pred,_=adapter(x,position_embeddings=rope)
            nmse,cos=local_metrics(pred,y)
            loss=nmse+.2*(1-cos)
        if not bool(torch.isfinite(loss)):raise FloatingPointError((variant,step,float(loss)))
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params,1.)
        scaler.step(opt);scaler.update();sched.step()
        if step==1 or step==STEPS or step%max(1,STEPS//8)==0:
            result=validate_local(adapter)
            score=result['nmse']+.2*(1-result['cosine'])
            if score<best_loss:
                best_loss=score;best_report=result
                best_sd={k:v.detach().cpu().clone() for k,v in adapter.core.state_dict().items()}
            print(f'{variant} {step}/{STEPS}: cosine={result["cosine"]:.5f} '
                  f'NMSE={result["nmse"]:.5f}',flush=True)
    assert best_sd is not None
    adapter.core.load_state_dict(best_sd);adapter.eval()
    return adapter,best_report,best_sd


## 8 · Evaluate native and candidate Qwen

Compute next-token negative log-likelihood and top-1 agreement on **all eligible positions**, and use paired GPU timing at the same context length. Do not describe a reduced cosine error as a language-model win without checking next-token behavior.


In [ ]:
@torch.inference_mode()
def measure_lm(blocks,reference=None):
    count=0;loss_total=0.;correct=0;agreement=0;top1=[]
    for begin in range(0,len(blocks),BATCH_SIZE):
        tok=blocks[begin:begin+BATCH_SIZE]
        idx=torch.arange(tok.shape[1]-1,device=DEVICE)
        with torch.autocast('cuda',dtype=DTYPE):
            logits=model(input_ids=tok.to(DEVICE),use_cache=False,
                         logits_to_keep=idx).logits.float()
        gold=tok[:,1:].to(DEVICE)
        logp=F.log_softmax(logits,dim=-1)
        loss_total-=logp.gather(-1,gold.unsqueeze(-1)).sum().item()
        pred=logits.argmax(-1)
        correct+=(pred==gold).sum().item();count+=gold.numel()
        if reference is None:top1.append(pred.detach().cpu().to(torch.int32))
        else:
            ref=reference[begin//BATCH_SIZE].to(DEVICE)
            assert ref.shape==pred.shape
            agreement+=(ref==pred).sum().item()
    result=dict(nll=loss_total/count,accuracy=correct/count,decisions=count)
    if reference is not None:result['teacher_agreement']=agreement/count
    return result,top1

@torch.inference_mode()
def paired_timing(adapter,blocks,warmups=3,repeats=12):
    sample=blocks[:min(len(blocks),BATCH_SIZE)].to(DEVICE)
    native_times=[];student_times=[]
    def once(use_student):
        with graft(adapter) if use_student else nullcontext():
            begin=torch.cuda.Event(enable_timing=True)
            end=torch.cuda.Event(enable_timing=True)
            begin.record()
            with torch.autocast('cuda',dtype=DTYPE):
                logits=model(input_ids=sample,use_cache=False,logits_to_keep=1).logits
            end.record();torch.cuda.synchronize()
            assert bool(torch.isfinite(logits).all())
            return begin.elapsed_time(end)
    torch.cuda.synchronize()
    for turn in range(warmups+repeats):
        if turn%2: st=once(True);nt=once(False)
        else: nt=once(False);st=once(True)
        if turn>=warmups:native_times.append(nt);student_times.append(st)
    return dict(native_ms=float(np.median(native_times)),student_ms=float(np.median(student_times)),
                speedup=float(np.median(native_times)/np.median(student_times)),
                native_iqr_ms=float(np.subtract(*np.percentile(native_times,[75,25]))),
                student_iqr_ms=float(np.subtract(*np.percentile(student_times,[75,25]))),
                repeats=repeats)

NATIVE_DEV,NATIVE_DEV_TOP1=measure_lm(DEV)
print('Native validation',NATIVE_DEV)


## 9 · Compare all variants on validation

All five candidates start from the automatically calibrated V5-style baseline checkpoint, with no interactive inputs. Save every candidate checkpoint, metrics, provenance and the baseline SHA-256.


In [ ]:
REPORT=[]
RUN_SPEC=dict(model_revision=REVISION,git_sha=GIT_SHA,code_sha256=MODULE_SHA256,
              baseline_checkpoint_sha256=V5_CHECKPOINT_SHA256,baseline_fingerprint=BOOTSTRAP_FINGERPRINT,
              baseline_steps=BASELINE_STEPS,baseline_variant=BOOTSTRAP_VARIANT,
              profile=PROFILE,seed=SEED,steps=STEPS,task_steps=TASK_STEPS,seq_len=SEQ_LEN,
              audit_offset=AUDIT_START_TOKEN,lr=LR,rank=RANK,candidates=CANDIDATES)
FINGERPRINT=hashlib.sha256(json.dumps(RUN_SPEC,sort_keys=True).encode()).hexdigest()
for variant in CANDIDATES:
    print('\n==========',variant,'==========')
    adapter,local,sd=fit(variant)
    with graft(adapter): dev,_=measure_lm(DEV,reference=NATIVE_DEV_TOP1)
    timing=paired_timing(adapter,DEV,repeats=5,warmups=2)
    row=dict(variant=variant,layer=TARGET_LAYER,steps=0 if variant=='v5_frozen' else STEPS,
             local_dev_cosine=local['cosine'],local_dev_nmse=local['nmse'],
             dev_nll=dev['nll'],dev_delta_nll=dev['nll']-NATIVE_DEV['nll'],
             dev_agreement=dev['teacher_agreement'],decisions=dev['decisions'],
             parameters=sum(p.numel() for p in adapter.core.parameters()),
             paired_speedup=timing['speedup'],native_ms=timing['native_ms'],
             student_ms=timing['student_ms'])
    torch.save(dict(core_state=sd,variant=variant,run_fingerprint=FINGERPRINT,
                    source_model_revision=REVISION,baseline_variant=BOOTSTRAP_VARIANT,
                    core_config=adapter.core.config),
               OUT/'checkpoints'/f'v52_L{TARGET_LAYER}_{variant}.pt')
    REPORT.append(row)
    pd.DataFrame(REPORT).to_csv(OUT/'validation_candidates.csv',index=False)
    print(json.dumps(row,indent=2),flush=True)
    del adapter,sd;gc.collect();torch.cuda.empty_cache()
print('Completed',len(REPORT),'candidates')


## 10 · Predeclared acceptance and validation-only ranking

Acceptance requires cosine ≥0.95, teacher agreement ≥0.98 and ΔNLL ≤0.10. If none pass, select the strongest *diagnostic* candidate without deploying it. Validation timing is a screen, not a production speedup claim.


In [ ]:
MIN_COSINE=.95
MIN_AGREEMENT=.98
MAX_DELTA_NLL=.10
ranked=pd.DataFrame(REPORT)
ranked['quality_gate']=((ranked.local_dev_cosine>=MIN_COSINE)&
                        (ranked.dev_agreement>=MIN_AGREEMENT)&
                        (ranked.dev_delta_nll<=MAX_DELTA_NLL))
ranked=ranked.sort_values(['quality_gate','dev_delta_nll','local_dev_nmse'],
                         ascending=[False,True,True]).reset_index(drop=True)
display(ranked[['variant','local_dev_cosine','local_dev_nmse','dev_delta_nll',
                'dev_agreement','paired_speedup','quality_gate']])
eligible=ranked[ranked.quality_gate]
if len(eligible):
    winner=eligible.sort_values(['paired_speedup','dev_delta_nll'],ascending=[False,True]).iloc[0]
    accepted=True
else:
    winner=ranked.iloc[0]
    accepted=False
print('Selected diagnostic:',winner.variant,'quality accepted:',accepted)
ranked.to_csv(OUT/'validation_ranked.csv',index=False)

## 11 · Optional whole-model KL refinement on **training inputs only**

The previous run showed good local cosine but insufficient Qwen next-token decision agreement. This experiment optionally fine-tunes the **joint V5.2 candidate** against native teacher logits on training blocks. Only validation data is used for selecting the refined checkpoint; the held-out diagnostic tail is not consulted until selection is complete. KL refinement costs additional GPU time and may still fail.


In [ ]:
# @title Optional end-to-end distillation
ENABLE_TASK_KL=True # @param {type:'boolean'}
if ENABLE_TASK_KL:
    name='v52_joint_w128'
    path=OUT/'checkpoints'/f'v52_L{TARGET_LAYER}_{name}.pt'
    payload=torch.load(path,map_location='cpu',weights_only=True)
    assert payload['run_fingerprint']==FINGERPRINT
    adapter=build_candidate(name)
    adapter.core.load_state_dict(payload['core_state'])
    trainable=set_trainables(adapter,name)
    opt=torch.optim.AdamW(trainable,lr=2e-5,weight_decay=.001)
    rng=np.random.default_rng(SEED+3)
    positions=torch.arange(max(0,SEQ_LEN-33),SEQ_LEN-1,2,device=DEVICE)
    for step in range(TASK_STEPS):
        choices=torch.tensor(rng.choice(len(TRAIN),BATCH_SIZE,replace=False))
        batch=TRAIN[choices].to(DEVICE)
        with torch.no_grad(),torch.autocast('cuda',dtype=DTYPE):
            teacher_logits=model(input_ids=batch,use_cache=False,
                                 logits_to_keep=positions).logits.float()
        opt.zero_grad(set_to_none=True)
        with graft(adapter),torch.autocast('cuda',dtype=DTYPE):
            student_logits=model(input_ids=batch,use_cache=False,
                                 logits_to_keep=positions).logits.float()
        tlog=F.log_softmax(teacher_logits,dim=-1).detach()
        slog=F.log_softmax(student_logits,dim=-1)
        loss=F.kl_div(slog,tlog.exp(),reduction='none').sum(-1).mean()
        assert bool(torch.isfinite(loss))
        loss.backward();torch.nn.utils.clip_grad_norm_(trainable,1.)
        opt.step()
        if (step+1)%max(1,TASK_STEPS//4)==0:print('task KL',step+1,float(loss))
    adapter.eval()
    local=validate_local(adapter)
    with graft(adapter):dev,_=measure_lm(DEV,reference=NATIVE_DEV_TOP1)
    timing=paired_timing(adapter,DEV,repeats=5,warmups=2)
    new_row=dict(variant='v52_joint_w128_plus_kl',layer=TARGET_LAYER,steps=STEPS,
                 local_dev_cosine=local['cosine'],local_dev_nmse=local['nmse'],
                 dev_nll=dev['nll'],dev_delta_nll=dev['nll']-NATIVE_DEV['nll'],
                 dev_agreement=dev['teacher_agreement'],decisions=dev['decisions'],
                 parameters=sum(p.numel() for p in adapter.core.parameters()),
                 paired_speedup=timing['speedup'],native_ms=timing['native_ms'],
                 student_ms=timing['student_ms'])
    REPORT.append(new_row)
    torch.save(dict(core_state={k:v.detach().cpu().clone() for k,v in adapter.core.state_dict().items()},
                    variant='v52_joint_w128_plus_kl',run_fingerprint=FINGERPRINT,
                    source_model_revision=REVISION,baseline_variant=BOOTSTRAP_VARIANT,
                    core_config=adapter.core.config),
               OUT/'checkpoints'/f'v52_L{TARGET_LAYER}_v52_joint_w128_plus_kl.pt')
    ranked=pd.DataFrame(REPORT)
    ranked['quality_gate']=((ranked.local_dev_cosine>=MIN_COSINE)&
                            (ranked.dev_agreement>=MIN_AGREEMENT)&
                            (ranked.dev_delta_nll<=MAX_DELTA_NLL))
    ranked=ranked.sort_values(['quality_gate','dev_delta_nll','local_dev_nmse'],
                              ascending=[False,True,True]).reset_index(drop=True)
    passing=ranked[ranked.quality_gate]
    if len(passing):
        winner=passing.sort_values(['paired_speedup','dev_delta_nll'],ascending=[False,True]).iloc[0]
        accepted=True
    else:
        winner=ranked.iloc[0];accepted=False
    ranked.to_csv(OUT/'validation_ranked.csv',index=False)
    print('Updated selected candidate',winner.variant,'accepted?',accepted)
else:
    print('End-to-end KL skipped; validation selection remains unchanged.')


## 12 · Diagnostic audit (new WikiText-2 training-tail offset)

The earlier experiments already inspected WikiText-2 test and a train tail near token 250,000. Here we audit on a disjoint train tail beginning at token **1,000,000**. It is a new diagnostic partition relative to these specific notebook samples, **not an independent external test set and not a guarantee of generalization**. Do not tune based on audit results.


In [ ]:
SELECTED=str(winner.variant)
checkpoint=OUT/'checkpoints'/f'v52_L{TARGET_LAYER}_{SELECTED}.pt'
payload=torch.load(checkpoint,map_location='cpu',weights_only=True)
assert payload['run_fingerprint']==FINGERPRINT
# Restore the actual trained variant and its router/correction flags.
student=build_candidate(SELECTED)
student.core.load_state_dict(payload['core_state']);student.eval()
AUDIT=build_windows('train',AUDIT_BLOCKS,SEQ_LEN,start_token=AUDIT_START_TOKEN)
FINGERPRINTS['audit_train_tail']=hashlib.sha256(AUDIT.numpy().tobytes()).hexdigest()
assert len(set(FINGERPRINTS.values()))==3
NATIVE_AUDIT,NATIVE_AUDIT_TOP1=measure_lm(AUDIT)
with graft(student):STUDENT_AUDIT,_=measure_lm(AUDIT,reference=NATIVE_AUDIT_TOP1)
PAIRED_AUDIT=paired_timing(student,AUDIT,repeats=20,warmups=5)
DELTA_NLL=STUDENT_AUDIT['nll']-NATIVE_AUDIT['nll']
FINAL_PASS=bool(accepted and DELTA_NLL<=MAX_DELTA_NLL and
                STUDENT_AUDIT['teacher_agreement']>=MIN_AGREEMENT and
                PAIRED_AUDIT['speedup']>1.05)
FINAL=dict(architecture='CeNN V5.2 zero-init output correction + larger local retrieval',
           model=MODEL_ID,model_revision=REVISION,repo_sha=GIT_SHA,
           baseline_origin='trained_automatically_in_this_notebook_no_upload',
           baseline_variant=BOOTSTRAP_VARIANT,baseline_steps=BASELINE_STEPS,
           baseline_fingerprint=BOOTSTRAP_FINGERPRINT,
           baseline_checkpoint_sha256=V5_CHECKPOINT_SHA256,code_sha256=MODULE_SHA256,
           profile=PROFILE,run_fingerprint=FINGERPRINT,seed=SEED,
           selected=SELECTED,validation_accepted=accepted,final_pass=FINAL_PASS,
           acceptance=dict(min_cosine=MIN_COSINE,min_agreement=MIN_AGREEMENT,
                           max_delta_nll=MAX_DELTA_NLL,min_speedup=1.05),
           validation=ranked.to_dict('records'),
           native_audit=NATIVE_AUDIT,student_audit=STUDENT_AUDIT,
           audit_delta_nll=DELTA_NLL,paired_audit=PAIRED_AUDIT,
           fingerprints=FINGERPRINTS,
           provenance='Diagnostic audit = disjoint WikiText-2 train tail, offset 1000000 tokens; '
                      'NOT the previously consulted WikiText-2 test split',
           limitations='Full-prefix CUDA prefill only; no Qwen hybrid KV cache, variable padding, '
                       'multimodal model, fused recurrent kernel or cross-model replication.')
(OUT/'final_audit.json').write_text(json.dumps(FINAL,indent=2,allow_nan=False))
print(json.dumps({key:FINAL[key] for key in
    ['selected','validation_accepted','final_pass','native_audit','student_audit',
     'audit_delta_nll','paired_audit']},indent=2))
if not FINAL_PASS:print('REJECT: retain native Qwen full attention. No speed/quality claim.')


## 13 · Context-length benchmark (optional)

Paired context timing can be enabled for 128–2,048 tokens, but the reference recurrent kernel is not fused and no hybrid cache integration exists. Do not claim autoregressive decoding speedup.


In [ ]:
# @title Optional length scaling
RUN_LENGTH_SWEEP=False # @param {type:'boolean'}
SWEEP=[]
if RUN_LENGTH_SWEEP:
    for length in [128,256,512,1024,2048]:
        try:
            sample=build_windows('validation',4,length)
            timing=paired_timing(student,sample,repeats=10,warmups=4)
            SWEEP.append(dict(length=length,**timing))
            print('Length',length,timing)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache();print('OOM at',length,'stopping');break
    pd.DataFrame(SWEEP).to_csv(OUT/'context_length_timing.csv',index=False)
    FINAL['length_sweep']=SWEEP
    (OUT/'final_audit.json').write_text(json.dumps(FINAL,indent=2,allow_nan=False))

## 14 · Export all results automatically

A complete ZIP of V5 auto-bootstrap and all V5.1 candidates, audit JSON, metrics and plot is generated at the end. **No upload is required**; the ZIP is only a downloadable output.


In [ ]:
import shutil
frame=pd.DataFrame(REPORT)
fig,ax=plt.subplots(figsize=(11,4))
ax.barh(frame['variant'],frame['local_dev_cosine'])
ax.axvline(MIN_COSINE,ls='--',color='gray',label='cosine gate 0.95')
ax.set(xlabel='Validation teacher-mixer cosine',xlim=(0,1),
       title='Qwen3.5 CeNN V5.2 — fidelity recovery')
ax.legend();fig.tight_layout();fig.savefig(OUT/'cosine_comparison.png',dpi=160)
(OUT/'manifest.json').write_text(json.dumps(dict(
    model=MODEL_ID,revision=REVISION,git_sha=GIT_SHA,
    notebook='Qwen35_08B_CeNN_HeadRescue_V52_FidelityRecovery_Colab.ipynb',
    profile=PROFILE,baseline_origin='automatic_training_no_upload',baseline_variant=BOOTSTRAP_VARIANT,
    baseline_steps=BASELINE_STEPS,baseline_fingerprint=BOOTSTRAP_FINGERPRINT,
    baseline_checkpoint_sha256=V5_CHECKPOINT_SHA256,source_code_sha256=MODULE_SHA256,
    candidates=list(CANDIDATES),audit_split='WikiText-2 train tail after 1000000 tokens',
    quality_acceptance=accepted,final_pass=FINAL_PASS),indent=2))
archive=shutil.make_archive(str(OUT),'zip',root_dir=OUT)
print('Result archive:',archive)
from IPython.display import display,FileLink
display(FileLink(archive))


### Interpretation and research limits

- If window 128 beats window 64 with the same router/post parameters, the longer local retrieval horizon is beneficial at this data/sequence length; measure its additional memory/runtime cost.
- If `v52_post_w64` beats V5.1 router-only, an output-conditioned residual adds useful information. A query-only correction is not proof of associative long-context recovery.
- If KL improves teacher agreement without raising NLL or hurting cosine, larger-scale teacher alignment becomes justified; the current quick split is still small.
- **Do not accept a student** unless cosine, agreement, NLL and **paired GPU latency** gates pass, including the diagnostic audit. Native SDPA remains the safe default.
- The current wrapper does not support Qwen's hybrid cached decoding, padding, vision inputs, multi-GPU partitioning or fused recurrent kernels.

Outputs: `validation_candidates.csv`, `validation_ranked.csv`, `final_audit.json`, manifest, plot and checkpoints in `/content/qwen35_cenn_head_rescue_v52.zip`.
